In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from PIL import Image

In [ ]:
dataset_path="/content/drive/MyDrive/CASIA2"

In [ ]:
import os

dataset_path = "/content/drive/MyDrive/CASIA2"

au_path = os.path.join(dataset_path, "Au")
tp_path = os.path.join(dataset_path, "Tp")

valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

image_paths = []
labels = []

def load_images(folder, label):
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.lower().endswith(valid_extensions):
                image_paths.append(os.path.join(root, file))
                labels.append(label)

# Load authentic images
load_images(au_path, 0)

# Load tampered images
load_images(tp_path, 1)

print("Total valid images:", len(image_paths))

Total valid images: 12614


In [ ]:
for path in image_paths:
    if path.endswith(".txt"):
        print(path)

In [ ]:
train_paths, test_paths, train_labels, test_labels = train_test_split(
    image_paths, labels, test_size=0.3, stratify=labels, random_state=42
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    test_paths, test_labels, test_size=0.5, stratify=test_labels, random_state=42
)

print("Train:", len(train_paths))
print("Validation:", len(val_paths))
print("Test:", len(test_paths))

Train: 8829
Validation: 1892
Test: 1893


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
class CASIADataset(Dataset):

    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        img_path = self.image_paths[idx]
        label = self.labels[idx]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
train_dataset = CASIADataset(train_paths, train_labels, transform)
val_dataset = CASIADataset(val_paths, val_labels, transform)
test_dataset = CASIADataset(test_paths, test_labels, transform)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels[:10])

torch.Size([32, 3, 224, 224])
tensor([0, 1, 1, 0, 1, 0, 0, 0, 0, 1])


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
model = models.vgg16(pretrained=True)

# Replace final classifier layer
model.classifier[6] = nn.Linear(4096, 2)

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 187MB/s]


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
EPOCHS = 15

train_losses = []
val_losses = []

train_acc = []
val_acc = []

In [ ]:
for epoch in range(EPOCHS):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = correct / total

    train_losses.append(train_loss)
    train_acc.append(train_accuracy)

    # ------------------
    # Validation
    # ------------------

    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = torch.max(outputs,1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = running_loss / len(val_loader)
    val_accuracy = correct / total

    val_losses.append(val_loss)
    val_acc.append(val_accuracy)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("Train Loss:", train_loss)
    print("Train Accuracy:", train_accuracy)
    print("Validation Loss:", val_loss)
    print("Validation Accuracy:", val_accuracy)

100%|██████████| 276/276 [36:23<00:00,  7.91s/it]



Epoch 1/15
Train Loss: 0.5847367051502933
Train Accuracy: 0.6869407633933627
Validation Loss: 0.5232902154326439
Validation Accuracy: 0.7436575052854123


100%|██████████| 276/276 [03:33<00:00,  1.29it/s]



Epoch 2/15
Train Loss: 0.4930530540321184
Train Accuracy: 0.764186204553177
Validation Loss: 0.5115330477555593
Validation Accuracy: 0.7383720930232558


100%|██████████| 276/276 [03:26<00:00,  1.34it/s]


...


Epoch 13/15
Train Loss: 0.23723136074841022
Train Accuracy: 0.8802808925133084
Validation Loss: 0.511329014847676
Validation Accuracy: 0.7743128964059197


100%|██████████| 276/276 [03:27<00:00,  1.33it/s]



Epoch 14/15
Train Loss: 0.2294169507542814
Train Accuracy: 0.8831124702684335
Validation Loss: 0.6816127908726534
Validation Accuracy: 0.7891120507399577


100%|██████████| 276/276 [03:27<00:00,  1.33it/s]



Epoch 15/15
Train Loss: 0.22656746471867614
Train Accuracy: 0.8887756257786839
Validation Loss: 0.8840393404165904
Validation Accuracy: 0.7700845665961945


In [ ]:
model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        y_true.extend(labels.numpy())
        y_pred.extend(predicted.cpu().numpy())

In [ ]:
accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(y_true, y_pred)

recall = recall_score(y_true, y_pred)

f1 = f1_score(y_true, y_pred)

print("\nTest Metrics")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


Test Metrics
Accuracy: 0.7776016904384575
Precision: 0.7148148148148148
Recall: 0.752925877763329
F1 Score: 0.7333755541481951


In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,6))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

plt.title("Confusion Matrix")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
plt.figure()

plt.plot(train_acc, label="Train Accuracy")
plt.plot(val_acc, label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Epoch")

plt.legend()

plt.show()

In [ ]:
plt.figure()

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss vs Epoch")

plt.legend()

plt.show()

In [ ]:
torch.save(model.state_dict(), "vgg16_casia_model.pth")

In [ ]:
torch.save(model.state_dict(),
"/content/drive/MyDrive/vgg16_casia_model.pth")